# Milestone A — Checkpoint 2
## Skema Entitas dan Pedoman Anotasi

Notebook ini menjalankan validasi skema Pydantic, mengekspor JSON Schema, memeriksa contoh positif dan negatif, serta membaca laporan audit Checkpoint 2.

In [2]:
from __future__ import annotations

import json
import subprocess
import sys
from collections import Counter
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "Riset").exists() else cwd.parent
if not (REPO_ROOT / "Riset").exists():
    raise FileNotFoundError("Root repositori tidak ditemukan.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from rag.id_entity_schema import ClinicalExtractionDocument, Stage2InputDocument

SCRIPT = REPO_ROOT / "Riset/scripts/checkpoint_02_validate_schema.py"
CHECKPOINT_01_INPUT = REPO_ROOT / "Riset/Tahap_2/checkpoint_01/stage2_checkpoint_01_input.jsonl"
EXAMPLES_FILE = REPO_ROOT / "Riset/Tahap_2/checkpoint_02/stage2_schema_examples.jsonl"
INVALID_EXAMPLES_FILE = REPO_ROOT / "Riset/Tahap_2/checkpoint_02/stage2_schema_invalid_examples.json"
GUIDELINE_FILE = REPO_ROOT / "Riset/Tahap_2/checkpoint_02/stage2_annotation_guideline.md"
INPUT_SCHEMA_FILE = REPO_ROOT / "Riset/Tahap_2/checkpoint_02/stage2_input_document_schema.json"
ENTITY_SCHEMA_FILE = REPO_ROOT / "Riset/Tahap_2/checkpoint_02/stage2_entity_extraction_schema.json"
AUDIT_FILE = REPO_ROOT / "Riset/Tahap_2/checkpoint_02/stage2_checkpoint_02_audit.json"

print("Repository:", REPO_ROOT)
print("Script    :", SCRIPT)
print("Examples  :", EXAMPLES_FILE)
print("Guideline :", GUIDELINE_FILE)

Repository: D:\Disertasi_Pipeline\cde_mapper
Script    : D:\Disertasi_Pipeline\cde_mapper\Riset\scripts\checkpoint_02_validate_schema.py
Examples  : D:\Disertasi_Pipeline\cde_mapper\Riset\Tahap_2\checkpoint_02\stage2_schema_examples.jsonl
Guideline : D:\Disertasi_Pipeline\cde_mapper\Riset\Tahap_2\checkpoint_02\stage2_annotation_guideline.md


## 1. Validasi file yang diperlukan

In [3]:
required = [SCRIPT, CHECKPOINT_01_INPUT, EXAMPLES_FILE, INVALID_EXAMPLES_FILE, GUIDELINE_FILE]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"File tidak ditemukan: {missing}")
print("Semua file tersedia.")

Semua file tersedia.


## 2. Jalankan skrip Checkpoint 2

In [4]:
command = [
    sys.executable,
    str(SCRIPT),
    "--checkpoint-01-input", str(CHECKPOINT_01_INPUT),
    "--examples-file", str(EXAMPLES_FILE),
    "--invalid-examples-file", str(INVALID_EXAMPLES_FILE),
    "--guideline-file", str(GUIDELINE_FILE),
    "--input-schema-file", str(INPUT_SCHEMA_FILE),
    "--entity-schema-file", str(ENTITY_SCHEMA_FILE),
    "--audit-file", str(AUDIT_FILE),
]
result = subprocess.run(
    command,
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
    check=False,
)
print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(f"Checkpoint gagal dengan exit code {result.returncode}")

{
  "status": "pass",
  "checkpoint_01_valid_documents": 300,
  "positive_examples": 9,
  "negative_examples_rejected": 6,
  "covered_entity_types": 10,
  "missing_core_entity_types": [],
  "audit_file": "D:\\Disertasi_Pipeline\\cde_mapper\\Riset\\Tahap_2\\checkpoint_02\\stage2_checkpoint_02_audit.json"
}



## 3. Baca laporan audit

In [5]:
audit = json.loads(AUDIT_FILE.read_text(encoding="utf-8"))

print("Status                    :", audit["status"])
print("Dokumen Checkpoint 1 valid:", audit["checkpoint_01_validation"]["valid_documents"])
print("Contoh positif            :", audit["positive_examples"]["total"])
print("Contoh negatif ditolak    :", audit["negative_examples"]["correctly_rejected"])
print("Entity type tercakup      :", len(audit["coverage"]["covered_entity_types"]))
print("Entity type belum tercakup:", audit["coverage"]["missing_core_entity_types"])
print("Pedoman baris             :", audit["guideline"]["lines"])

Status                    : pass
Dokumen Checkpoint 1 valid: 300
Contoh positif            : 9
Contoh negatif ditolak    : 6
Entity type tercakup      : 10
Entity type belum tercakup: []
Pedoman baris             : 464


## 4. Periksa exit criteria

In [6]:
for criterion, passed in audit["exit_criteria"].items():
    marker = "PASS" if passed else "FAIL"
    print(f"[{marker}] {criterion}")

assert audit["status"] == "pass"
assert all(audit["exit_criteria"].values())

[PASS] checkpoint_01_all_documents_valid
[PASS] positive_examples_available
[PASS] all_positive_examples_valid
[PASS] all_negative_examples_rejected
[PASS] core_entity_types_covered
[PASS] question_and_answer_examples_available
[PASS] assertion_examples_cover_context
[PASS] experiencer_examples_cover_patient_and_family
[PASS] guideline_required_sections_available
[PASS] json_schemas_exported


## 5. Validasi langsung contoh positif

In [7]:
validated_examples = []
for line in EXAMPLES_FILE.read_text(encoding="utf-8").splitlines():
    raw = json.loads(line)
    example_id = raw.pop("example_id")
    document = ClinicalExtractionDocument.model_validate(raw)
    validated_examples.append((example_id, document))

print(f"{len(validated_examples)} contoh positif valid.")
for example_id, document in validated_examples:
    print(example_id, "->", len(document.entities), "entitas")

9 contoh positif valid.
example-q1-negation -> 1 entitas
example-q2-family -> 1 entitas
example-q3-measurement -> 2 entitas
example-q4-drug -> 1 entitas
example-q5-procedure-anatomy -> 2 entitas
example-q6-demographic-visit -> 3 entitas
example-q7-clinical-finding -> 2 entitas
example-a1-differential -> 2 entitas
example-a2-recommendation -> 2 entitas


## 6. Tampilkan cakupan label

In [8]:
entity_types = Counter()
assertions = Counter()
temporals = Counter()
experiencers = Counter()
epistemics = Counter()

for _, document in validated_examples:
    for entity in document.entities:
        entity_types[entity.entity_type] += 1
        assertions[entity.assertion] += 1
        temporals[entity.temporal] += 1
        experiencers[entity.experiencer] += 1
        epistemics[entity.epistemic_status] += 1

print("Entity type:", dict(entity_types))
print("Assertion  :", dict(assertions))
print("Temporal   :", dict(temporals))
print("Experiencer:", dict(experiencers))
print("Epistemic  :", dict(epistemics))

Entity type: {'symptom': 3, 'condition': 2, 'measurement': 1, 'unit': 1, 'drug': 1, 'procedure': 2, 'anatomy': 2, 'demographic': 2, 'visit': 1, 'clinical_finding': 1}
Assertion  : {'negated': 1, 'present': 11, 'uncertain': 2, 'hypothetical': 2}
Temporal   : {'present': 9, 'unknown': 3, 'past': 2, 'future': 2}
Experiencer: {'patient': 15, 'family': 1}
Epistemic  : {'patient_fact': 12, 'general_information': 1, 'differential_diagnosis': 1, 'recommended': 1, 'conditional': 1}


## 7. Inspeksi JSON Schema

In [9]:
input_schema = json.loads(INPUT_SCHEMA_FILE.read_text(encoding="utf-8"))
entity_schema = json.loads(ENTITY_SCHEMA_FILE.read_text(encoding="utf-8"))

print("Input schema title :", input_schema.get("title"))
print("Entity schema title:", entity_schema.get("title"))
print("Definitions        :", sorted(entity_schema.get("$defs", {}).keys()))

Input schema title : Stage2InputDocument
Entity schema title: ClinicalExtractionDocument
Definitions        : ['Assertion', 'ClinicalDomain', 'ClinicalEntity', 'EntityType', 'EpistemicStatus', 'Experiencer', 'SourceField', 'Speaker', 'Temporal']


## 8. Pemeriksaan akhir

In [10]:
assert audit["checkpoint_01_validation"]["valid_documents"] == 300
assert not audit["positive_examples"]["failures"]
assert not audit["negative_examples"]["failures"]
assert not audit["coverage"]["missing_core_entity_types"]
assert not audit["guideline"]["missing_required_sections"]
assert INPUT_SCHEMA_FILE.exists()
assert ENTITY_SCHEMA_FILE.exists()

print("Checkpoint 2 berhasil dan seluruh exit criteria terpenuhi.")
print("Entity schema:", ENTITY_SCHEMA_FILE)
print("Laporan audit:", AUDIT_FILE)

Checkpoint 2 berhasil dan seluruh exit criteria terpenuhi.
Entity schema: D:\Disertasi_Pipeline\cde_mapper\Riset\Tahap_2\checkpoint_02\stage2_entity_extraction_schema.json
Laporan audit: D:\Disertasi_Pipeline\cde_mapper\Riset\Tahap_2\checkpoint_02\stage2_checkpoint_02_audit.json


## 9. Contoh input dan label entitas

Bagian ini menampilkan beberapa contoh input yang membentuk cakupan label pada Bagian 6. Setiap entitas ditampilkan bersama `entity_type`, `assertion`, `temporal`, `experiencer`, dan `epistemic_status`.

In [11]:
selected_example_ids = {
    "example-q1-negation",
    "example-q2-family",
    "example-q3-measurement",
    "example-q4-drug",
    "example-q7-clinical-finding",
    "example-a1-differential",
    "example-a2-recommendation",
}

selected_examples = [
    (example_id, document)
    for example_id, document in validated_examples
    if example_id in selected_example_ids
]

for example_id, document in selected_examples:
    print("=" * 110)
    print(f"Example      : {example_id}")
    print(f"Source       : {document.source_field} / {document.speaker}")
    print(f"Input        : {document.normalized_text}")
    print("-" * 110)
    print(
        f"{'MENTION':<28} {'ENTITY TYPE':<20} {'ASSERTION':<14} "
        f"{'TEMPORAL':<12} {'EXPERIENCER':<14} {'EPISTEMIC STATUS'}"
    )
    print("-" * 110)
    for entity in document.entities:
        print(
            f"{entity.mention:<28} {entity.entity_type:<20} "
            f"{entity.assertion:<14} {entity.temporal:<12} "
            f"{entity.experiencer:<14} {entity.epistemic_status}"
        )
    print()

assert len(selected_examples) == len(selected_example_ids)
print(f"Ditampilkan {len(selected_examples)} contoh input tervalidasi.")

Example      : example-q1-negation
Source       : question / patient
Input        : saya tidak demam sejak dua hari.
--------------------------------------------------------------------------------------------------------------
MENTION                      ENTITY TYPE          ASSERTION      TEMPORAL     EXPERIENCER    EPISTEMIC STATUS
--------------------------------------------------------------------------------------------------------------
demam                        symptom              negated        present      patient        patient_fact

Example      : example-q2-family
Source       : question / patient
Input        : ibu saya menderita diabetes melitus.
--------------------------------------------------------------------------------------------------------------
MENTION                      ENTITY TYPE          ASSERTION      TEMPORAL     EXPERIENCER    EPISTEMIC STATUS
--------------------------------------------------------------------------------------------------------